# Module 01 — Tabular Q-learning & SARSA

Watch a value function form from nothing but a NumPy table. Read [the lesson](README.md) alongside this notebook.

> Tip: in Colab, set **Runtime → Change runtime type → GPU**.

In [ ]:
# === Colab setup: run me first ===
import os, sys
if not os.path.exists('rl'):
    # On Colab, clone the repo so the `rl` package is importable.
    !git clone https://github.com/anhduckkzz/lunarlander.git repo && (cp -r repo/* . 2>/dev/null || true)
    !pip -q install 'gymnasium[box2d]>=0.29' torch numpy matplotlib imageio tqdm
import torch
print('Torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

## Q-learning on FrozenLake
The whole algorithm is one update rule. Train, then inspect the learned table.

In [ ]:
import gymnasium as gym
from rl.agents.tabular import TabularAgent, train_tabular

env = gym.make('FrozenLake-v1', is_slippery=False)
agent = TabularAgent(env.observation_space.n, env.action_space.n,
                     algo='q_learning', alpha=0.1, eps_decay=0.9995)
scores = train_tabular(env, agent, n_episodes=8000, log_every=1000)

### Inspect the learned value function

In [ ]:
import numpy as np
V = agent.Q.max(axis=1).reshape(4, 4)   # FrozenLake is a 4x4 grid
print('State values (max_a Q):')
print(np.round(V, 2))

## Now compare SARSA (on-policy) vs Q-learning on a slippery lake
Turn on stochastic transitions and see how the safer, on-policy SARSA behaves.

In [ ]:
import numpy as np
from rl.agents.tabular import TabularAgent, train_tabular
import gymnasium as gym
for algo in ['q_learning', 'sarsa']:
    env = gym.make('FrozenLake-v1', is_slippery=True)
    ag = TabularAgent(env.observation_space.n, env.action_space.n, algo=algo, alpha=0.1)
    sc = train_tabular(env, ag, n_episodes=20000, verbose=False)
    print(f'{algo:10s} final avg success: {np.mean(sc[-1000:]):.3f}')

**Exercise:** try `CliffWalking-v0`. Plot the path each agent prefers. Why does SARSA avoid the cliff edge?